In [ ]:
import pandas as pd
import numpy as np
import itertools
import joblib

In [ ]:
#CARICAMENTO MODELLI E SCALER
modello_apo = joblib.load('modello_apogeo_v2.pkl')
modello_ang = joblib.load('modello_angolo_v2.pkl')
scaler_fisso = joblib.load('scaler_v2.pkl')
interpolatori_rischio = joblib.load('motore_rischio_v2.pkl')

In [ ]:
#CARICAMENTO FUNZIONI 

def calcola_setup(vento_reale, std_reale, apogeo_obiettivo, modello_apo, modello_ang, scaler):
    lista_zavorre = [0, 20, 40, 60]
    delay_possibili = np.arange(0.8, 3.01, 0.05) 
    
    if vento_reale < 0.2:
        rampe_possibili = [0.0]
    else:
        angolo_min = vento_reale * -5.0
        angolo_max = vento_reale * -3.0
        rampe_possibili = np.arange(angolo_min, angolo_max + 0.1, 0.1)

    combinazioni = list(itertools.product([vento_reale], [std_reale], lista_zavorre, delay_possibili, rampe_possibili))
    df = pd.DataFrame(combinazioni, columns=['vento_medio', 'std_vento', 'zavorra', 'delay_secondi', 'angolo_rampa'])
    
    dati_scalati = scaler.transform(df)
    
    df['Prev_Apogeo'] = modello_apo.predict(dati_scalati)
    df['Prev_Errore_Inclinazione'] = modello_ang.predict(dati_scalati)
    
    ottimali = []
    tolleranza_metri = 5.0
    tolleranza_angolo_sicuro = 1.0 
    
    for z in lista_zavorre:
        df_z = df[df['zavorra'] == z].copy()
        
        df_z_target = df_z[np.abs(df_z['Prev_Apogeo'] - apogeo_obiettivo) <= tolleranza_metri]
        
        if not df_z_target.empty:
            migliore = df_z_target.iloc[np.abs(df_z_target['Prev_Errore_Inclinazione']).argsort()[:1]].iloc[0]
            ottimali.append(migliore)
            
        else:
            df_z_dritti = df_z[np.abs(df_z['Prev_Errore_Inclinazione']) <= tolleranza_angolo_sicuro]
            
            if not df_z_dritti.empty:
                distanza_dal_target = np.abs(df_z_dritti['Prev_Apogeo'] - apogeo_obiettivo)
                migliore_ripiego = df_z_dritti.iloc[distanza_dal_target.argsort()[:1]].iloc[0]
                ottimali.append(migliore_ripiego)

    if not ottimali:
        return "ALLARME ROSSO: Nessun setup sicuro o in target trovato per nessuna zavorra."
        
    df_finale = pd.DataFrame(ottimali)
    df_finale = df_finale.round({'delay_secondi': 2, 'angolo_rampa': 1, 'Prev_Apogeo': 1, 'Prev_Errore_Inclinazione': 1})
    
    return df_finale[['zavorra', 'angolo_rampa', 'delay_secondi', 'Prev_Apogeo', 'Prev_Errore_Inclinazione']]

#FUNZIONE PER STIMARE IL RISCHIO DI UN SETUP NON SIMULATO 
def stima_rischio(v, std, z):
    modello = interpolatori_rischio.get(z)
    
    b = modello['bias'](v, std)
    s = modello['std'](v, std)
    w_min = modello['min'](v, std)
    w_max = modello['max'](v, std)
    
    return float(b), float(s), float(w_min), float(w_max)

In [ ]:
def genera_report_missione(vento_live, std_live, apogeo, mod_apo, mod_ang, scaler):
    
    #CHIAMA IL MODELLO DI OTTIMIZZAZIONE
    df_setup = calcola_setup(vento_reale=vento_live, std_reale=std_live, apogeo_obiettivo=apogeo, modello_apo=mod_apo, modello_ang=mod_ang, scaler=scaler)
    
    if isinstance(df_setup, str):
        return df_setup
        
    risultati = []
    
    #CHIAMA IL MODELLO DEL RISCHIO
    for _, riga in df_setup.iterrows():
        z = riga['zavorra']
        errore_modello = riga['Prev_Errore_Inclinazione']

        #ESTRAZIONE DATI DI RISCHIO
        bias, sigma, w_min, w_max = stima_rischio(vento_live, std_live, z)
        
        angolo_atteso = 90.0 - (errore_modello + bias)
        angolo_estremo_1 = 90.0 - w_max
        angolo_estremo_2 = 90.0 - w_min
        
        finestra_min = min(angolo_estremo_1, angolo_estremo_2)
        finestra_max = max(angolo_estremo_1, angolo_estremo_2)
            
        risultati.append({
            'Zavorra_g': z,
            'Rampa_gradi': riga['angolo_rampa'],
            'Delay_s': riga['delay_secondi'],
            'Apogeo_Atteso_m': riga['Prev_Apogeo'],
            'Delta_Target_m': round(riga['Prev_Apogeo'] - apogeo, 1),
            'ANG_ATTESO_ACC_C6': round(angolo_atteso, 1),
            'Peggior_Caso_+': round(finestra_max, 1),
            'Peggior_Caso_-': round(finestra_min, 1),
            'Dispersione_Std': round(sigma, 2)})

    df_finale = pd.DataFrame(risultati)
    
    nome_file = f"Report_Lancio_V{vento_live}_T{std_live}_Apo{apogeo}.csv"
    df_finale.to_csv(nome_file, index=False)
    
    return df_finale

In [ ]:
#ESECUZIONE SUL CAMPO
VENTO_ATTUALE = 3.5
STD_ATTUALE = 0.5
APOGEO = 180

report = genera_report_missione(VENTO_ATTUALE, STD_ATTUALE, APOGEO, modello_apo, modello_ang, scaler_fisso)
print(report.to_string(index=False))